# 1. Import libraries and set the random seed

In [ ]:
import math
import random
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Use a single CPU thread so this tiny model runs quickly in classroom environments.
torch.set_num_threads(1)

# Set the random seed for reproducible results.
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


# 2. Prepare a small text classification dataset

In [ ]:
# This is a tiny toy dataset.
# Label 1 means positive sentiment, and label 0 means negative sentiment.
raw_data = [
    ("i love this movie", 1),
    ("this film is great", 1),
    ("what a fantastic story", 1),
    ("the acting was wonderful", 1),
    ("i really enjoyed this film", 1),
    ("this movie made me happy", 1),
    ("the story was touching", 1),
    ("i like this actor", 1),
    ("i hate this movie", 0),
    ("this film is terrible", 0),
    ("what a boring story", 0),
    ("the acting was awful", 0),
    ("i really disliked this film", 0),
    ("this movie made me angry", 0),
    ("the story was dull", 0),
    ("i do not like this actor", 0),
]

random.shuffle(raw_data)

texts = [text for text, label in raw_data]
labels = [label for text, label in raw_data]

print("Number of samples:", len(texts))
print("First three samples:")
for text, label in raw_data[:3]:
    print(f"label={label}, text={text}")

Number of samples: 16
First three samples:
label=1, text=i like this actor
label=0, text=this film is terrible
label=1, text=this movie made me happy


# 3. Tokenization

In [ ]:
def tokenize(sentence):
    # Convert the sentence to lowercase and split it by whitespace.
    return sentence.lower().split()

tokenized_texts = [tokenize(text) for text in texts]

print("Before tokenization:", texts[0])
print("After tokenization :", tokenized_texts[0])

Before tokenization: i like this actor
After tokenization : ['i', 'like', 'this', 'actor']


# 4. Build the vocabulary

In [ ]:
# Count token frequencies in the training corpus.
counter = Counter()
for tokens in tokenized_texts:
    counter.update(tokens)

# Reserve 0 for padding and 1 for unknown tokens.
word_to_index = {"<PAD>": 0, "<UNK>": 1}

for word, count in counter.most_common():
    word_to_index[word] = len(word_to_index)

index_to_word = {index: word for word, index in word_to_index.items()}

vocab_size = len(word_to_index)

print("Vocabulary size:", vocab_size)
print("Token-to-index dictionary:")
print(word_to_index)

Vocabulary size: 34
Token-to-index dictionary:
{'<PAD>': 0, '<UNK>': 1, 'this': 2, 'i': 3, 'film': 4, 'movie': 5, 'the': 6, 'story': 7, 'was': 8, 'like': 9, 'actor': 10, 'is': 11, 'made': 12, 'me': 13, 'what': 14, 'a': 15, 'really': 16, 'acting': 17, 'terrible': 18, 'happy': 19, 'touching': 20, 'dull': 21, 'boring': 22, 'disliked': 23, 'hate': 24, 'great': 25, 'fantastic': 26, 'angry': 27, 'do': 28, 'not': 29, 'enjoyed': 30, 'awful': 31, 'love': 32, 'wonderful': 33}


# 5. Integer encoding and padding

In [ ]:
def texts_to_sequences(tokenized_texts, word_to_index):
    # Convert each token to its integer ID.
    encoded_texts = []
    for tokens in tokenized_texts:
        encoded = [word_to_index.get(token, word_to_index["<UNK>"]) for token in tokens]
        encoded_texts.append(encoded)
    return encoded_texts

encoded_texts = texts_to_sequences(tokenized_texts, word_to_index)

print("Original tokens:", tokenized_texts[0])
print("Integer-encoded tokens:", encoded_texts[0])

Original tokens: ['i', 'like', 'this', 'actor']
Integer-encoded tokens: [3, 9, 2, 10]


In [ ]:
def pad_sequences(sequences, max_len, pad_value=0):
    # Pad each sequence to the same length.
    padded_sequences = np.full((len(sequences), max_len), pad_value, dtype=np.int64)

    for i, sequence in enumerate(sequences):
        sequence = sequence[:max_len]
        padded_sequences[i, :len(sequence)] = sequence

    return padded_sequences

max_len = max(len(sequence) for sequence in encoded_texts)
padded_texts = pad_sequences(encoded_texts, max_len=max_len, pad_value=word_to_index["<PAD>"])

X = torch.tensor(padded_texts, dtype=torch.long)
y = torch.tensor(labels, dtype=torch.long)

print("Maximum sequence length:", max_len)
print("Input tensor shape:", X.shape)
print("Label tensor shape:", y.shape)
print("First padded sample:", X[0])

Maximum sequence length: 6
Input tensor shape: torch.Size([16, 6])
Label tensor shape: torch.Size([16])
First padded sample: tensor([ 3,  9,  2, 10,  0,  0])


# 6. Create DataLoader objects

In [ ]:
# Split the tiny dataset into training and test sets.
train_size = int(len(X) * 0.75)

X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

print("Training set size:", len(train_dataset))
print("Test set size:", len(test_dataset))

Training set size: 12
Test set size: 4


# 7. Positional encoding

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()

        # pe has shape (max_len, d_model).
        pe = torch.zeros(max_len, d_model)

        # position has shape (max_len, 1).
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        # div_term controls the wavelength of each dimension.
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        # Apply sine to even dimensions and cosine to odd dimensions.
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # Add a batch dimension: (1, max_len, d_model).
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        # x has shape (batch_size, seq_len, d_model).
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len, :]

# 8. Build a small Transformer Encoder classifier

In [ ]:
class SmallTransformerClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model=16,
        nhead=2,
        num_layers=1,
        dim_feedforward=32,
        num_classes=2,
        pad_idx=0,
        dropout=0.1,
    ):
        super().__init__()

        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.positional_encoding = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )

        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
        )

        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, input_ids):
        # input_ids has shape (batch_size, seq_len).
        padding_mask = (input_ids == self.pad_idx)

        x = self.embedding(input_ids)

        # The embedding output is scaled in the original Transformer paper.
        x = x * math.sqrt(x.size(-1))

        x = self.positional_encoding(x)

        encoded = self.transformer_encoder(
            x,
            src_key_padding_mask=padding_mask,
        )

        # Mean pooling over non-padding tokens.
        mask = (~padding_mask).unsqueeze(-1).float()
        summed = (encoded * mask).sum(dim=1)
        lengths = mask.sum(dim=1).clamp(min=1)
        pooled = summed / lengths

        logits = self.classifier(pooled)
        return logits


model = SmallTransformerClassifier(
    vocab_size=vocab_size,
    d_model=16,
    nhead=2,
    num_layers=1,
    dim_feedforward=32,
    num_classes=2,
    pad_idx=word_to_index["<PAD>"],
).to(device)

print(model)

SmallTransformerClassifier(
  (embedding): Embedding(34, 16, padding_idx=0)
  (positional_encoding): PositionalEncoding()
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0): TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=16, out_features=16, bias=True)
        )
        (linear1): Linear(in_features=16, out_features=32, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=32, out_features=16, bias=True)
        (norm1): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (classifier): Linear(in_features=16, out_features=2, bias=True)
)


# 9. Train the model

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

num_epochs = 30

for epoch in range(1, num_epochs + 1):
    model.train()
    total_loss = 0.0

    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        logits = model(batch_X)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if epoch % 10 == 0:
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch:03d} | Training loss: {avg_loss:.4f}")

Epoch 010 | Training loss: 0.4183
Epoch 020 | Training loss: 0.0903
Epoch 030 | Training loss: 0.0527


# 10. Evaluate the model

In [ ]:
def evaluate(model, data_loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_X, batch_y in data_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            logits = model(batch_X)
            predictions = torch.argmax(logits, dim=1)

            correct += (predictions == batch_y).sum().item()
            total += batch_y.size(0)

    accuracy = correct / total
    return accuracy

train_accuracy = evaluate(model, train_loader)
test_accuracy = evaluate(model, test_loader)

print(f"Training accuracy: {train_accuracy:.4f}")
print(f"Test accuracy    : {test_accuracy:.4f}")

Training accuracy: 1.0000
Test accuracy    : 0.5000


# 11. Run inference on new sentences

In [ ]:
def predict_sentiment(sentence, model, word_to_index, max_len):
    model.eval()

    tokens = tokenize(sentence)
    encoded = [word_to_index.get(token, word_to_index["<UNK>"]) for token in tokens]
    padded = pad_sequences([encoded], max_len=max_len, pad_value=word_to_index["<PAD>"])

    input_ids = torch.tensor(padded, dtype=torch.long).to(device)

    with torch.no_grad():
        logits = model(input_ids)
        probabilities = torch.softmax(logits, dim=1)
        prediction = torch.argmax(probabilities, dim=1).item()

    label_name = "positive" if prediction == 1 else "negative"
    return label_name, probabilities.squeeze(0).cpu().numpy()


test_sentences = [
    "i love this actor",
    "this movie was awful",
    "what a wonderful film",
    "i do not like this story",
]

for sentence in test_sentences:
    label, probabilities = predict_sentiment(sentence, model, word_to_index, max_len)
    print(f"Input sentence: {sentence}")
    print(f"Predicted label: {label}")
    print(f"Class probabilities [negative, positive]: {probabilities}")
    print()

Input sentence: i love this actor
Predicted label: positive
Class probabilities [negative, positive]: [0.00517572 0.9948243 ]

Input sentence: this movie was awful
Predicted label: positive
Class probabilities [negative, positive]: [0.4594638  0.54053617]

Input sentence: what a wonderful film
Predicted label: negative
Class probabilities [negative, positive]: [0.9598178 0.0401822]

Input sentence: i do not like this story
Predicted label: negative
Class probabilities [negative, positive]: [0.97654086 0.0234591 ]

